In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from gen_variable_standard_static import \
    metrics_search_for_two_fragments_df
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source',
       'num_days_actual_records', 'sample_year'],
      dtype='str')

In [3]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [4]:
ac_power_metrics = metrics_search_for_two_fragments_df(metrics_df, 'ac', 'pow', 'and')
ac_power_systems = set(ac_power_metrics['system_id'])
two_years_days = 2 * 365
enough_data_systems = systems_cleaned[systems_cleaned['num_days_actual_records'] >= two_years_days]
enough_data_ids = set(enough_data_systems.system_id)
enough_data_parquet_power_systems = enough_data_ids.intersection(ac_power_systems)
enough_data_parquet_power_list = list(enough_data_parquet_power_systems)
enough_data_parquet_power_list.sort()

In [5]:
systems_cleaned_enough_ac_data = systems_cleaned[systems_cleaned['system_id'].isin(enough_data_parquet_power_systems)]

In [6]:
systems_cleaned_enough_ac_data.loc[:, 'location'] = systems_cleaned_enough_ac_data.apply(
    lambda row: (row['latitude'], row['longitude']),
    axis = 1
)

In [7]:
relevant_locs = systems_cleaned_enough_ac_data[['latitude', 'longitude']].drop_duplicates()
relevant_locs.shape

(50, 2)

In [8]:
fig = px.scatter_geo(
    data_frame = relevant_locs,
    lat = 'latitude',
    lon = 'longitude'
)

fig.update_layout(
        title = 'Locations of PVDAQ good-data sites',
        # comment or un-comment to get global vs. USA
        geo_scope='usa',
    )
fig.write_html('../../../results/eda/trimmedmap.html')
fig.show()

In [9]:
relevant_locs_b = systems_cleaned_enough_ac_data[['system_id', 'latitude', 'longitude', 'kg_climate']].drop_duplicates()
relevant_locs_b.shape

(82, 4)

In [10]:
fig = px.scatter_geo(
    data_frame = relevant_locs_b,
    lat = 'latitude',
    lon = 'longitude',
    color = 'system_id'
)

fig.update_layout(
        title = 'Locations of PVDAQ good-data sites',
        # comment or un-comment to get global vs. USA
        geo_scope='usa',
    )
# fig.write_html('../../../results/eda/trimmedmap.html')
fig.show()

Literal overlaps

In [63]:
good_system_locs = systems_cleaned_enough_ac_data.groupby(by=['latitude', 'longitude', 'location'])['system_id'].agg(['unique', 'count'])
good_system_locs = good_system_locs.reset_index()
good_system_locs = good_system_locs.sort_values(by=['longitude'])
good_system_locs['count'] = good_system_locs['count'].astype('str')  # for graphing purposes

In [64]:
fig = px.scatter_geo(
    data_frame = good_system_locs,
    lat = 'latitude',
    lon = 'longitude',
    color = 'count'
)

fig.update_layout(
        title = 'Locations of PVDAQ good-data sites',
        # comment or un-comment to get global vs. USA
        geo_scope='usa',
    )
# fig.write_html('../../../results/eda/trimmedmap.html')
fig.show()

## Colorado cluster

In [11]:
colorado_sites = systems_cleaned_enough_ac_data[
    (systems_cleaned_enough_ac_data['longitude'] > -106)
    & (systems_cleaned_enough_ac_data['longitude'] < -105)
]

In [12]:
colorado_sites[['latitude', 'longitude', 'elevation_m']].describe()

,latitude,longitude,elevation_m
count,16.000000,16.000000,16.000000
mean,39.749063,-105.178119,1819.662500
std,0.043749,0.014407,71.324964
min,39.704000,-105.231100,1770.000000
25%,39.740400,-105.177400,1780.000000
50%,39.740550,-105.175300,1793.850000
75%,39.741700,-105.172500,1807.750000
max,39.909400,-105.169400,1994.700000


In [13]:
colorado_sites.head()

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year,location
2,4,NREL x-Si -1,"Golden, CO",7,39.7406,-105.1774,1795.3,1.00,BSk,12,...,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,5063,2008,"(39.7406, -105.1774)"
3,10,NREL CIS -1,"Golden, CO",7,39.7404,-105.1774,1792.8,1.12,BSk,12,...,True,True,True,True,cis family thin-film,thin_film,PVDAQ General,5893,2007,"(39.7404, -105.1774)"
4,33,Silicor Materials,"Golden, CO",7,39.7404,-105.1772,1794.0,2.40,BSk,12,...,True,True,True,True,Unknown,unknown,PVDAQ General,4438,2011,"(39.7404, -105.1772)"
7,36,NREL low-X x-Si -1,"Golden, CO",7,39.7040,-105.1773,1794.7,2.21,BSk,12,...,True,True,True,True,Unknown,unknown,PVDAQ General,2516,2013,"(39.704, -105.1773)"
8,50,NREL x-Si 6,"Golden, CO",7,39.7420,-105.1727,1994.7,6.00,BSk,12,...,True,True,True,True,Unknown,unknown,PVDAQ General,8455,1995,"(39.742, -105.1727)"


In [14]:
from geopy.distance import distance

In [15]:
colorado_sites.loc[:, 'location'] = colorado_sites.apply(
    lambda row: (row['latitude'], row['longitude']),
    axis=1
)

In [16]:
site_50_loc = colorado_sites.at[8, 'location']
site_4_loc = colorado_sites.at[2, 'location']

In [17]:
colorado_sites.loc[:, 'distance_to_50'] = colorado_sites.apply(
    lambda row: distance(row['location'], site_50_loc),
    axis = 1
)

In [18]:
colorado_sites[['system_id', 'distance_to_50']].iloc[0:10]

,system_id,distance_to_50
2,4,0.43180962070123097 km
3,10,0.4402914586198599 km
4,33,0.42466205639177373 km
7,36,4.237507444101983 km
8,50,0.0 km
9,51,0.07464879130388567 km
17,1208,0.31755856889583584 km
68,1283,0.18364334889300948 km
69,1284,0.4359297910582695 km
70,1289,0.4359297910582695 km


In [19]:
colorado_sites.loc[:, 'distance_to_4'] = colorado_sites.apply(
    lambda row: distance(row['location'], site_4_loc),
    axis = 1
)

In [20]:
colorado_sites[['system_id', 'distance_to_4']].iloc[10:20]

,system_id,distance_to_4
107,1332,0.41176477332484107 km
108,1332,0.41176477332484107 km
150,1430,0.3578704422274643 km
151,1431,19.29769290876161 km
152,1432,0.5037934197567435 km
153,1433,0.4719619397989635 km


In [21]:
colorado_sites[colorado_sites['distance_to_50'] > 4]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year,location,distance_to_50,distance_to_4
7,36,NREL low-X x-Si -1,"Golden, CO",7,39.7040,-105.1773,1794.7,2.21,BSk,12,...,True,True,Unknown,unknown,PVDAQ General,2516,2013,"(39.704, -105.1773)",4.237507444101983 km,4.063681454371833 km
151,1431,[1431] NREL Windsite PV 1-axis,"Boulder, CO",America/Denver,39.9094,-105.2311,1850.0,1082.85,Dfb,11,...,True,True,ribbon polycrystalline si,multicrystalline_Si,PVDAQ General,2957,2010,"(39.9094, -105.2311)",19.2473271520153 km,19.29769290876161 km


In [22]:
colorado_sites[['distance_to_4', 'distance_to_50']].describe()

,distance_to_4,distance_to_50
count,16,16
unique,14,14
top,0.011102965088901625 km,0.4359297910582695 km
freq,2,2


In [23]:
fig = px.scatter_geo(
    data_frame = colorado_sites,
    lat = 'latitude',
    lon = 'longitude',
    color = 'elevation_m'
)

fig.update_layout(
        title = 'Locations of PVDAQ good-data sites',
        # comment or un-comment to get global vs. USA
        geo_scope='usa',
    )
# fig.write_html('../../../results/eda/trimmedmap.html')
fig.show()

In [24]:
from itertools import product

In [25]:
colorado_close_sites = colorado_sites[colorado_sites['distance_to_4'] < 4]
close_site_locs = colorado_close_sites['location'].unique()
max_dist = 0
for loc_1, loc_2 in product(close_site_locs, repeat=2):
    dist = distance(loc_1, loc_2)
    if dist > max_dist:
        max_dist = dist
max_dist

Distance(0.8057801641593175)

In [26]:
colorado_close_sites.system_id.nunique()

13

In [27]:
nevada_cluster = systems_cleaned_enough_ac_data[
    (systems_cleaned_enough_ac_data['longitude'] > -115.5)
    & (systems_cleaned_enough_ac_data['longitude'] < -114.5)
]

In [28]:
nevada_cluster[['latitude', 'longitude']].describe()

,latitude,longitude
count,12.000000,12.000000
mean,36.128633,-115.087492
std,0.075305,0.113689
min,36.027500,-115.271300
25%,36.033000,-115.158200
50%,36.159400,-115.155800
75%,36.195200,-114.975825
max,36.195200,-114.921500


In [29]:
nevada_cluster.system_id.nunique()

12

In [30]:
nevada_location_names = set(nevada_cluster.site_location.unique())

In [31]:
def diam_2D(locations):
    max_dist = 0
    for loc_1, loc_2 in product(locations, repeat=2):
        dist = distance(loc_1, loc_2)
        if dist > max_dist:
            max_dist = dist
    return max_dist


In [32]:
nevada_locations = nevada_cluster['location'].unique()
diam_2D(nevada_locations)

Distance(33.32518133036213)

In [33]:
dist_dict = {
    name: 0 for name in nevada_location_names
}
for name in nevada_location_names:
    this_site_loc = nevada_cluster[nevada_cluster['site_location'] == name]
    this_site_coords = this_site_loc['location'].unique()
    print(f'{name} has {this_site_loc.system_id.nunique()} many systems')
    dist_dict[name] = diam_2D(this_site_coords)

Las Vegas, NV has 5 many systems
Henderson, NV has 4 many systems
Clark County, NV has 3 many systems


In [34]:
dist_dict

{'Las Vegas, NV': 0,
 'Henderson, NV': Distance(5.6310080563504386),
 'Clark County, NV': Distance(22.332905575997184)}

In [35]:
fig = px.scatter_geo(
    data_frame = nevada_cluster,
    lat = 'latitude',
    lon = 'longitude',
    color = 'system_id'
)

fig.update_layout(
        title = 'Locations of PVDAQ good-data sites in Nevada',
        # comment or un-comment to get global vs. USA
        geo_scope='usa',
    )
# fig.write_html('../../../results/eda/trimmedmap.html')
fig.show()

In [36]:
nevada_cluster[nevada_cluster['system_id'] == 1278]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year,location
67,1278,Andre Agassi Preparatory Academy - Building D,"Las Vegas, NV",America/Los_Angeles,36.1952,-115.1582,620.0,171.36,Bwh,26,...,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,3254,2012,"(36.1952, -115.1582)"


In [37]:
nevada_cluster[nevada_cluster['system_id'] == 1420]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year,location
146,1420,[1420] Clark County-NV - Government Center,"Clark County, NV",America/Los_Angeles,36.1654,-115.1534,678.0,30.0,Bwh,26,...,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,1802,2016,"(36.1654, -115.1534)"


In [38]:
distance(nevada_cluster.at[67, 'location'], nevada_cluster.at[146, 'location'])

Distance(3.3347516800055335)

In [39]:
nevada_cluster[nevada_cluster['system_id'].isin((1369, 1419))]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year,location
141,1369,[1369] City of Henderson-NV - Police Station,"Henderson, NV",America/Los_Angeles,36.0300,-114.9839,660.0,NaN,Bwh,26,...,True,True,True,True,Unknown,unknown,PVDAQ General,2509,2014,"(36.03, -114.9839)"
145,1419,[1419] Clark County-NV - Hollywood Rec Center,"Clark County, NV",America/Los_Angeles,36.1534,-115.0256,779.0,40.0,Bwh,26,...,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,1781,2016,"(36.1534, -115.0256)"


In [40]:
distance(nevada_cluster.at[141, 'location'], nevada_cluster.at[145, 'location'])

Distance(14.198220607123782)

In [41]:
nevada_cluster[nevada_cluster['site_location'] == 'Henderson, NV']

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year,location
139,1367,[1367] City of Henderson-NV - Aquatic Complex,"Henderson, NV",America/Los_Angeles,36.0330,-114.9516,660.0,277.16,Bwh,26,...,False,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,2219,2014,"(36.033, -114.9516)"
140,1368,[1368] City of Henderson-NV - Heritage Park,"Henderson, NV",America/Los_Angeles,36.0330,-114.9516,660.0,NaN,Bwh,26,...,True,True,True,True,Unknown,unknown,PVDAQ General,2614,2014,"(36.033, -114.9516)"
141,1369,[1369] City of Henderson-NV - Police Station,"Henderson, NV",America/Los_Angeles,36.0300,-114.9839,660.0,NaN,Bwh,26,...,True,True,True,True,Unknown,unknown,PVDAQ General,2509,2014,"(36.03, -114.9839)"
148,1423,[1423] RTC-NV-Baseline,"Henderson, NV",America/Los_Angeles,36.0275,-114.9215,697.0,6.00,Bwh,26,...,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,2186,2016,"(36.0275, -114.9215)"


In [42]:
louisiana_cluster = systems_cleaned_enough_ac_data[
    (systems_cleaned_enough_ac_data['longitude'] > -91)
    & (systems_cleaned_enough_ac_data['longitude'] < -90)
]

In [43]:
louisiana_cluster.system_id.nunique()

25

In [44]:
louisiana_cluster.location.nunique()

7

In [45]:
fig = px.scatter_geo(
    data_frame = louisiana_cluster,
    lat = 'latitude',
    lon = 'longitude',
)

fig.update_layout(
        title = 'Locations of PVDAQ good-data sites in Nevada',
        # comment or un-comment to get global vs. USA
        geo_scope='usa',
    )
# fig.write_html('../../../results/eda/trimmedmap.html')
fig.show()

In [46]:
louisiana_cluster_locs = louisiana_cluster.groupby(['latitude', 'longitude'])['system_id'].agg(['unique', 'count'])

In [47]:
louisiana_cluster_locs = louisiana_cluster_locs.reset_index()
louisiana_cluster_locs = louisiana_cluster_locs.sort_values('longitude')

In [48]:
louisiana_cluster_locs

,latitude,longitude,unique,count
3,29.9745,-90.1563,[1249],1
0,29.9288,-90.1273,"[1256, 1273]",2
6,30.0017,-90.0926,"[1244, 1246, 1254, 1257, 1258, 1274]",6
1,29.9510,-90.0812,"[1248, 1250, 1252, 1253, 1272]",5
2,29.9649,-90.0690,[1275],1
5,29.9922,-90.0495,[1255],1
4,29.9809,-90.0023,"[1261, 1263, 1264, 1265, 1266, 1268, 1269, 127...",9


In [49]:
fig = px.scatter_geo(
    data_frame = louisiana_cluster_locs,
    lat = 'latitude',
    lon = 'longitude',
    text='count',
    opacity=0
)

fig.update_layout(
        title = 'Counts of PVDAQ good-data sites in New Orleans',
        # comment or un-comment to get global vs. USA
        geo_scope='usa',
    )
# fig.write_html('../../../results/eda/trimmedmap.html')
fig.show()

In [50]:
for j in range(7):
    dist_name = f'dist_to_site_{j}'
    key_loc = (louisiana_cluster_locs.at[j, 'latitude'], louisiana_cluster_locs.at[j, 'longitude'])
    louisiana_cluster_locs.loc[:, dist_name] = louisiana_cluster_locs.apply(
        lambda row: distance((row['latitude'], row['longitude']), key_loc),
        axis = 1
    )


In [51]:
louisiana_cluster_locs = louisiana_cluster_locs[['latitude', 'longitude', 'unique', 'count', 'dist_to_site_3',
                                                 'dist_to_site_0', 'dist_to_site_6', 'dist_to_site_1',
                                                 'dist_to_site_2', 'dist_to_site_5', 'dist_to_site_4']]

In [52]:
louisiana_cluster_locs

,latitude,longitude,unique,count,dist_to_site_3,dist_to_site_0,dist_to_site_6,dist_to_site_1,dist_to_site_2,dist_to_site_5,dist_to_site_4
3,29.9745,-90.1563,[1249],1,0.0 km,5.787961729558389 km,6.846590893197679 km,7.702698713971832 km,8.492746657995482 km,10.491556871103539 km,14.879130420537289 km
0,29.9288,-90.1273,"[1256, 1273]",2,5.787961729558389 km,0.0 km,8.747661829251706 km,5.085737421548597 km,6.905790766404045 km,10.285278545886483 km,13.377184441857553 km
6,30.0017,-90.0926,"[1244, 1246, 1254, 1257, 1258, 1274]",6,6.846590893197679 km,8.747661829251706 km,0.0 km,5.726873671713612 km,4.672043011008339 km,4.289951330599961 km,9.013376250602345 km
1,29.9510,-90.0812,"[1248, 1250, 1252, 1253, 1272]",5,7.702698713971832 km,5.085737421548597 km,5.726873671713612 km,0.0 km,1.9393282820128912 km,5.497168556637704 km,8.305389644354399 km
2,29.9649,-90.0690,[1275],1,8.492746657995482 km,6.905790766404045 km,4.672043011008339 km,1.9393282820128912 km,0.0 km,3.563672116134835 km,6.677249792286416 km
5,29.9922,-90.0495,[1255],1,10.491556871103539 km,10.285278545886483 km,4.289951330599961 km,5.497168556637704 km,3.563672116134835 km,0.0 km,4.723873283347367 km
4,29.9809,-90.0023,"[1261, 1263, 1264, 1265, 1266, 1268, 1269, 127...",9,14.879130420537289 km,13.377184441857553 km,9.013376250602345 km,8.305389644354399 km,6.677249792286416 km,4.723873283347367 km,0.0 km


## Tampa Cluster

In [53]:
tampa_cluster = systems_cleaned_enough_ac_data[
    (systems_cleaned_enough_ac_data['latitude'] > 27)
    & (systems_cleaned_enough_ac_data['latitude'] < 28)
    & (systems_cleaned_enough_ac_data['longitude'] > -83)
    & (systems_cleaned_enough_ac_data['longitude'] < -82)
]

In [54]:
tampa_cluster[['latitude', 'longitude']].value_counts()

latitude  longitude
27.7701   -82.6290     2
27.7936   -82.7144     2
27.7589   -82.6700     2
27.8175   -82.6517     1
27.8050   -82.6270     1
27.7883   -82.6414     1
27.7743   -82.6423     1
27.7643   -82.6497     1
Name: count, dtype: int64

In [55]:
tampa_cluster['system_id'].nunique()

11

In [56]:
tampa_cluster_locs = tampa_cluster.groupby(by=['latitude', 'longitude', 'location'])['system_id'].agg(['unique', 'count'])
tampa_cluster_locs = tampa_cluster_locs.reset_index()
tampa_cluster_locs = tampa_cluster_locs.sort_values(by=['longitude'])
tampa_cluster_locs

,latitude,longitude,location,unique,count
5,27.7936,-82.7144,"(27.7936, -82.7144)","[1307, 1326]",2
0,27.7589,-82.6700,"(27.7589, -82.67)","[1310, 1329]",2
7,27.8175,-82.6517,"(27.8175, -82.6517)",[1308],1
1,27.7643,-82.6497,"(27.7643, -82.6497)",[1325],1
3,27.7743,-82.6423,"(27.7743, -82.6423)",[1321],1
4,27.7883,-82.6414,"(27.7883, -82.6414)",[1318],1
2,27.7701,-82.6290,"(27.7701, -82.629)","[1300, 1319]",2
6,27.8050,-82.6270,"(27.805, -82.627)",[1314],1


In [59]:
tampa_cluster_locs.index

Index([5, 0, 7, 1, 3, 4, 2, 6], dtype='int64')

In [57]:
tampa_cluster_locs['count'] = tampa_cluster_locs['count'].astype('str')

In [58]:
fig = px.scatter_geo(
    data_frame = tampa_cluster_locs,
    lat = 'latitude',
    lon = 'longitude',
    color = 'count'
)

fig.update_layout(
        title = 'Locations of PVDAQ good-data sites in Tampa, FL',
        # comment or un-comment to get global vs. USA
        geo_scope='usa',
    )
# fig.write_html('../../../results/eda/trimmedmap.html')
fig.show()

In [60]:
for j in tampa_cluster_locs.index:
    col_name_j = f'dist_to_site_{j}'
    jth_loc = tampa_cluster_locs.at[j, 'location']
    tampa_cluster_locs.loc[:, col_name_j] = tampa_cluster_locs.apply(
        lambda row: distance(row['location'], jth_loc),
        axis=1
    )



In [61]:
tampa_cluster_locs

,latitude,longitude,location,unique,count,dist_to_site_5,dist_to_site_0,dist_to_site_7,dist_to_site_1,dist_to_site_3,dist_to_site_4,dist_to_site_2,dist_to_site_6
5,27.7936,-82.7144,"(27.7936, -82.7144)","[1307, 1326]",2,0.0 km,5.8256340327071205 km,6.722075291159491 km,7.155970458810415 km,7.420866781843538 km,7.218160015422196 km,8.810614372954548 km,8.704859607500893 km
0,27.7589,-82.6700,"(27.7589, -82.67)","[1310, 1329]",2,5.8256340327071205 km,0.0 km,6.739615582881896 km,2.088679333657808 km,3.2199113094780145 km,4.308278691041673 km,4.227848798159193 km,6.637692267809641 km
7,27.8175,-82.6517,"(27.8175, -82.6517)",[1308],1,6.722075291159491 km,6.739615582881896 km,0.0 km,5.898709795187611 km,4.876058344852234 km,3.391280555332873 km,5.709210994700822 km,2.8003522867204076 km
1,27.7643,-82.6497,"(27.7643, -82.6497)",[1325],1,7.155970458810415 km,2.088679333657808 km,5.898709795187611 km,0.0 km,1.3266760879575357 km,2.7825578998895923 km,2.1392844634394717 km,5.034603420484664 km
3,27.7743,-82.6423,"(27.7743, -82.6423)",[1321],1,7.420866781843538 km,3.2199113094780145 km,4.876058344852234 km,1.3266760879575357 km,0.0 km,1.5539568483634483 km,1.391122218772159 km,3.7212329099865045 km
4,27.7883,-82.6414,"(27.7883, -82.6414)",[1318],1,7.218160015422196 km,4.308278691041673 km,3.391280555332873 km,2.7825578998895923 km,1.5539568483634483 km,0.0 km,2.358255029527087 km,2.3320733287668007 km
2,27.7701,-82.6290,"(27.7701, -82.629)","[1300, 1319]",2,8.810614372954548 km,4.227848798159193 km,5.709210994700822 km,2.1392844634394717 km,1.391122218772159 km,2.358255029527087 km,0.0 km,3.8724991766348635 km
6,27.8050,-82.6270,"(27.805, -82.627)",[1314],1,8.704859607500893 km,6.637692267809641 km,2.8003522867204076 km,5.034603420484664 km,3.7212329099865045 km,2.3320733287668007 km,3.8724991766348635 km,0.0 km


In [62]:
tampa_cluster_locs['dist_to_site_4'].dtype

dtype('O')

## Eastern seaboard stretch